# Repository guide: MMS SpecAugment and speed perturbation

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


# MMS-1B → Tarifit V1.2 Adapter Fine-Tuning — SpecAugment + Speed Perturbation

This notebook is the third controlled MMS experiment.

It keeps the same frozen V1.2 corpus, tokenizer, MMS-1B base model, adapter-only training strategy,
learning rate, batch size, gradient accumulation, seed, validation set, greedy decoding, and 4 epochs.

The experimental change relative to the SpecAugment notebook is:

- SpecAugment time masking remains enabled.
- Training audio is randomly speed-perturbed on the fly with factors **0.9×, 1.0×, or 1.1×**.
- Validation audio is never speed-perturbed.

This tests whether combining feature-level augmentation with speaking-rate variation improves
generalization beyond SpecAugment alone.


In [ ]:
# Cell 1 — Install the exact reproducible dependencies

!pip -q install \
    "transformers==4.57.1" \
    "datasets==4.4.1" \
    "accelerate>=1.10,<2" \
    "jiwer==4.0.0" \
    "safetensors>=0.4.5" \
    "soundfile>=0.12.1"

print("✓ Dependencies installed.")
print("If Colab asks for a runtime restart after installation, restart once and continue from Cell 2.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
✓ Dependencies installed.
If Colab asks for a runtime restart after installation, restart once and continue from Cell 2

In [ ]:
# Cell 2 — Mount Google Drive and define combined-augmentation experiment paths

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

METADATA_PATH = PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2.csv"
FROZEN_METADATA_PATH = PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2_train_val_frozen.csv"

TOKENIZER_DIR = PROJECT_ROOT / "data" / "processed" / "mms_tokenizer_v1_2"
DATASET_CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "mms_corpus_v1_2"
CACHE_MANIFEST_PATH = DATASET_CACHE_DIR / "cache_manifest.json"

MODEL_DIR = PROJECT_ROOT / "models" / "mms_1b_tarifit_v1_2_specaug_speed"
RESULTS_DIR = PROJECT_ROOT / "results" / "mms_1b_tarifit_v1_2_specaug_speed"
TRAINER_STATE_DIR = MODEL_DIR / "trainer_state"

BEST_ADAPTER_PATH = MODEL_DIR / "best_adapter.safetensors"
BEST_ADAPTER_INFO_PATH = MODEL_DIR / "best_adapter_info.json"

NOAUG_SUMMARY_PATH = (
    PROJECT_ROOT / "results" / "mms_1b_tarifit_v1_2_noaug" / "experiment_summary.json"
)

SPECAUG_SUMMARY_PATH = (
    PROJECT_ROOT / "results" / "mms_1b_tarifit_v1_2_specaug" / "experiment_summary.json"
)

for path in [MODEL_DIR, RESULTS_DIR, TRAINER_STATE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Frozen metadata:", FROZEN_METADATA_PATH)
print("Shared dataset cache:", DATASET_CACHE_DIR)
print("Combined augmentation model output:", MODEL_DIR)
print("Combined augmentation results:", RESULTS_DIR)


Mounted at /content/drive
Project root: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm
Frozen metadata: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/segments_metadata_v1_2_train_val_frozen.csv
Shared dataset cache: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_corpus_v1_2
Combined augmentation model output: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_specaug_speed
Combined augmentation results: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_1b_tarifit_v1_2_specaug_speed


In [ ]:
# Cell 3 — Record software and GPU environment

import sys
import platform
import torch
import transformers
import datasets

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())

assert transformers.__version__ == "4.57.1", transformers.__version__
assert datasets.__version__ == "4.4.1", datasets.__version__

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("GPU runtime is required for MMS-1B fine-tuning.")

print("✓ Software versions match the cached V1.2 experiment environment.")


Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cu128
Transformers: 4.57.1
Datasets: 4.4.1
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 15.64
✓ Software versions match the cached V1.2 experiment environment.


In [ ]:
# Cell 4 — Load master metadata and recover the exact frozen 1754/129 split

import pandas as pd
import numpy as np

assert METADATA_PATH.exists(), f"Missing metadata: {METADATA_PATH}"
assert FROZEN_METADATA_PATH.exists(), (
    "Frozen V1.2 train/validation metadata is missing."
)

# ---------------------------------------------------------
# 1. Master metadata
# Keep this for speaker/test-leakage checks in later cells.
# ---------------------------------------------------------

df = pd.read_csv(METADATA_PATH)

for col in [
    "transcription",
    "final_selection",
    "review_status",
    "dataset_split",
]:
    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

df["final_selection"] = df["final_selection"].str.lower()
df["review_status"] = df["review_status"].str.lower()
df["dataset_split"] = df["dataset_split"].str.lower()


# ---------------------------------------------------------
# 2. Frozen metadata = source of truth for this experiment
# ---------------------------------------------------------

frozen_df = pd.read_csv(FROZEN_METADATA_PATH)

for col in [
    "transcription",
    "final_selection",
    "review_status",
    "dataset_split",
]:
    frozen_df[col] = (
        frozen_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

frozen_df["final_selection"] = frozen_df["final_selection"].str.lower()
frozen_df["review_status"] = frozen_df["review_status"].str.lower()
frozen_df["dataset_split"] = frozen_df["dataset_split"].str.lower()


train_selected = frozen_df[
    frozen_df["dataset_split"].eq("train")
].copy()

val_selected = frozen_df[
    frozen_df["dataset_split"].eq("validation")
].copy()

selected = pd.concat(
    [train_selected, val_selected],
    ignore_index=True,
)


# ---------------------------------------------------------
# 3. Exact frozen-split checks
# ---------------------------------------------------------

assert len(train_selected) == 1754, (
    f"Expected 1754 train segments, found {len(train_selected)}"
)

assert len(val_selected) == 129, (
    f"Expected 129 validation segments, found {len(val_selected)}"
)

assert len(selected) == 1883

assert not selected["segment_id"].duplicated().any(), (
    "Duplicate segment_id values found."
)


print("Frozen selected rows:", len(selected))
print("Train segments:", len(train_selected))
print("Validation segments:", len(val_selected))

print(
    "Train hours:",
    round(train_selected["duration_seconds"].sum() / 3600, 3)
)

print(
    "Validation hours:",
    round(val_selected["duration_seconds"].sum() / 3600, 3)
)

print(
    "Train speakers:",
    train_selected["speaker_group_id"].nunique()
)

print(
    "Validation speakers:",
    val_selected["speaker_group_id"].nunique()
)

print("\n✓ Exact frozen 1754/129 split recovered.")

Frozen selected rows: 1883
Train segments: 1754
Validation segments: 129
Train hours: 5.223
Validation hours: 0.298
Train speakers: 3
Validation speakers: 2

✓ Exact frozen 1754/129 split recovered.


In [ ]:
# Cell 5 — Verify speaker independence and test exclusion
train_speakers = set(train_selected["speaker_group_id"])
val_speakers = set(val_selected["speaker_group_id"])

overlap = train_speakers & val_speakers
print("Train speakers:", sorted(train_speakers))
print("Validation speakers:", sorted(val_speakers))
print("Train/validation speaker overlap:", overlap)
assert not overlap, f"Speaker leakage found: {overlap}"

selected_ids = set(selected["segment_id"])
test_rows = df[df["dataset_split"].eq("test")].copy()
test_ids = set(test_rows["segment_id"])

print("Selected train/validation rows also marked test:", len(selected_ids & test_ids))
assert not (selected_ids & test_ids), "Test segments leaked into train/validation."

print("✓ Speaker-independent train/validation split confirmed.")
print("✓ Test data is excluded from this experiment.")


Train speakers: ['SPK001', 'SPK002', 'SPK009']
Validation speakers: ['SPK007', 'SPK010']
Train/validation speaker overlap: set()
Selected train/validation rows also marked test: 0
✓ Speaker-independent train/validation split confirmed.
✓ Test data is excluded from this experiment.


In [ ]:
# Cell 6 — Validate the final V1.2 orthographic inventory
from collections import Counter
import unicodedata

FINAL_LETTERS = list("abcdefghijklmnpqrstuvwxyz") + ["ɛ", "ɣ", "ʷ", "ḍ", "ḥ", "ṭ"]
ALLOWED_CHARACTERS = set(FINAL_LETTERS) | {" "}

char_counts = Counter("".join(selected["transcription"].tolist()))
unexpected = {
    ch: count
    for ch, count in char_counts.items()
    if ch not in ALLOWED_CHARACTERS
}

print("Expected final letters:", " ".join(FINAL_LETTERS))
print("Number of expected letters:", len(FINAL_LETTERS))
print("Unexpected characters:", unexpected)

for text in selected["transcription"]:
    assert text == unicodedata.normalize("NFC", text), "Non-NFC transcription found."

assert not unexpected, f"Unexpected transcription characters remain: {unexpected}"
print("✓ V1.2 text matches the final normalized alphabet.")


Expected final letters: a b c d e f g h i j k l m n p q r s t u v w x y z ɛ ɣ ʷ ḍ ḥ ṭ
Number of expected letters: 31
Unexpected characters: {}
✓ V1.2 text matches the final normalized alphabet.


In [ ]:
# Cell 7 — Verify this experiment uses exactly the same frozen metadata as the no-augmentation run
import hashlib

freeze_columns = [
    "segment_id",
    "recording_id",
    "speaker_group_id",
    "dataset_split",
    "duration_seconds",
    "audio_path",
    "review_status",
    "transcription",
    "final_selection",
]

current_frozen = (
    selected[freeze_columns]
    .sort_values(["dataset_split", "segment_id"])
    .reset_index(drop=True)
)

existing_frozen = (
    pd.read_csv(FROZEN_METADATA_PATH)[freeze_columns]
    .sort_values(["dataset_split", "segment_id"])
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    current_frozen,
    existing_frozen,
    check_dtype=False,
)

sha256 = hashlib.sha256(FROZEN_METADATA_PATH.read_bytes()).hexdigest()

print("✓ Frozen metadata matches exactly.")
print("Frozen rows:", len(existing_frozen))
print("SHA256:", sha256)


✓ Frozen metadata matches exactly.
Frozen rows: 1883
SHA256: 4911f46e0a8c3656089677b8899d8296b9e1528d8aa3633a64728eb645f28fab


## Tokenizer

The tokenizer is not relearned for the augmentation run. It is reconstructed deterministically from the same final V1.2 alphabet and checked against the existing saved tokenizer.


In [ ]:
# Cell 8 — Load or verify the deterministic V1.2 CTC tokenizer
import json
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

vocab_tokens = ["|"] + FINAL_LETTERS + ["[UNK]", "[PAD]"]
expected_vocab = {token: idx for idx, token in enumerate(vocab_tokens)}
VOCAB_PATH = TOKENIZER_DIR / "vocab.json"

assert VOCAB_PATH.exists(), f"Missing tokenizer vocabulary: {VOCAB_PATH}"

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    existing_vocab = json.load(f)

assert existing_vocab == expected_vocab, (
    "Saved V1.2 tokenizer differs from the deterministic vocabulary."
)

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    bos_token=None,
    eos_token=None,
    do_lower_case=False,
)

BASE_MODEL_ID = "facebook/mms-1b-all"

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    BASE_MODEL_ID,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print("Tokenizer size:", len(tokenizer))
print("PAD/CTC blank id:", tokenizer.pad_token_id)
print("UNK id:", tokenizer.unk_token_id)
print("✓ Reusing the same V1.2 tokenizer.")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

Tokenizer size: 34
PAD/CTC blank id: 33
UNK id: 32
✓ Reusing the same V1.2 tokenizer.


In [ ]:
# Cell 9 — Verify all frozen references encode without UNK
unk_id = tokenizer.unk_token_id
unknown_segments = []

for row in existing_frozen.itertuples(index=False):
    ids = tokenizer(row.transcription).input_ids
    if unk_id in ids:
        unknown_segments.append((row.segment_id, row.transcription))

print("Segments containing [UNK]:", len(unknown_segments))

if unknown_segments:
    for item in unknown_segments[:20]:
        print(item)

assert not unknown_segments, "Some V1.2 references contain characters outside the tokenizer."
print("✓ All frozen references are fully representable.")


Segments containing [UNK]: 0
✓ All frozen references are fully representable.


## Dataset

This experiment reuses the exact processed dataset cache from the no-augmentation run. No augmented copies are written to disk because SpecAugment is applied internally by Wav2Vec2 only while the model is in training mode.


In [ ]:
# Cell 10 — Reuse the exact cached V1.2 dataset
from datasets import load_from_disk

assert DATASET_CACHE_DIR.exists(), f"Missing dataset cache: {DATASET_CACHE_DIR}"
assert CACHE_MANIFEST_PATH.exists(), f"Missing cache manifest: {CACHE_MANIFEST_PATH}"

with open(CACHE_MANIFEST_PATH, "r", encoding="utf-8") as f:
    cache_manifest = json.load(f)

assert cache_manifest.get("metadata_sha256") == sha256, (
    "Dataset cache hash does not match the frozen V1.2 metadata."
)
assert cache_manifest.get("tokenizer_size") == len(tokenizer), (
    "Dataset cache tokenizer size does not match the V1.2 tokenizer."
)

dataset = load_from_disk(str(DATASET_CACHE_DIR))
train_ds = dataset["train"]
val_ds = dataset["validation"]

assert len(train_ds) == 1754, f"Expected 1754 cached train examples, found {len(train_ds)}"
assert len(val_ds) == 129, f"Expected 129 cached validation examples, found {len(val_ds)}"

print("✓ Reused existing V1.2 dataset cache.")
print("Train examples:", len(train_ds))
print("Validation examples:", len(val_ds))
print("Dataset columns:", train_ds.column_names)


✓ Reused existing V1.2 dataset cache.
Train examples: 1754
Validation examples: 129
Dataset columns: ['segment_id', 'input_values', 'input_length', 'labels']


In [ ]:
# Cell 11 — Summarize the shared train/validation durations
def duration_summary(split_ds):
    seconds = np.array(split_ds["input_length"], dtype=np.float64) / 16000.0
    return {
        "segments": len(seconds),
        "hours": float(seconds.sum() / 3600),
        "min_s": float(seconds.min()),
        "max_s": float(seconds.max()),
        "mean_s": float(seconds.mean()),
    }

train_duration = duration_summary(train_ds)
val_duration = duration_summary(val_ds)

print("Train:", train_duration)
print("Validation:", val_duration)


Train: {'segments': 1754, 'hours': 5.222733593749999, 'min_s': 0.848, 'max_s': 19.984, 'mean_s': 10.71940760404789}
Validation: {'segments': 129, 'hours': 0.2983577777777778, 'min_s': 0.944, 'max_s': 26.0, 'mean_s': 8.326263565891473}


## Model initialization and augmentation

A fresh MMS-1B CTC head and language adapter are initialized exactly as in the no-augmentation experiment. The shared multilingual base remains frozen.

The only change is **SpecAugment time masking during training**. Feature masking is disabled to keep the augmentation mild and interpretable.


In [ ]:
# Cell 12 — Load MMS-1B and enable SpecAugment for the combined augmentation run

from transformers import Wav2Vec2ForCTC

SPEC_AUGMENT_CONFIG = {
    "apply_spec_augment": True,
    "mask_time_prob": 0.05,
    "mask_time_length": 5,
    "mask_time_min_masks": 1,
    "mask_feature_prob": 0.0,
}

SPEED_PERTURBATION_CONFIG = {
    "enabled": True,
    "factors": [0.9, 1.0, 1.1],
    "sampling": "uniform_random_per_training_example_access",
    "validation_augmented": False,
}

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL_ID,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    apply_spec_augment=SPEC_AUGMENT_CONFIG["apply_spec_augment"],
    mask_time_prob=SPEC_AUGMENT_CONFIG["mask_time_prob"],
    mask_time_length=SPEC_AUGMENT_CONFIG["mask_time_length"],
    mask_time_min_masks=SPEC_AUGMENT_CONFIG["mask_time_min_masks"],
    mask_feature_prob=SPEC_AUGMENT_CONFIG["mask_feature_prob"],
    ctc_loss_reduction="mean",
    pad_token_id=tokenizer.pad_token_id,
    vocab_size=len(tokenizer),
    ignore_mismatched_sizes=True,
)

model.init_adapter_layers()
model.freeze_base_model()

adapter_weights = model._get_adapters()
for parameter in adapter_weights.values():
    parameter.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")
print("SpecAugment enabled:", model.config.apply_spec_augment)
print("mask_time_prob:", model.config.mask_time_prob)
print("mask_time_length:", model.config.mask_time_length)
print("Speed factors:", SPEED_PERTURBATION_CONFIG["factors"])
print("Validation speed perturbation:", SPEED_PERTURBATION_CONFIG["validation_augmented"])

assert model.config.apply_spec_augment is True
assert trainable_params == 2_194_722, f"Unexpected trainable parameter count: {trainable_params}"

print("✓ MMS-1B combined-augmentation model initialized.")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([34]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([34, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 964,692,130
Trainable parameters: 2,194,722
Trainable percentage: 0.2275%
SpecAugment enabled: True
mask_time_prob: 0.05
mask_time_length: 5
Speed factors: [0.9, 1.0, 1.1]
Validation speed perturbation: False
✓ MMS-1B combined-augmentation model initialized.


In [ ]:
# Cell 13 — Confirm CTC feasibility, including the fastest 1.1× speed condition

def minimum_ctc_frames(labels):
    repeats = sum(
        labels[i] == labels[i - 1]
        for i in range(1, len(labels))
    )
    return len(labels) + repeats


def output_frames_for_samples(input_samples):
    return int(
        model._get_feat_extract_output_lengths(
            torch.tensor(int(input_samples))
        ).item()
    )


def find_ctc_infeasible(split_ds, speed_factor=1.0):
    bad = []

    for i, example in enumerate(split_ds):
        original_samples = int(example["input_length"])

        # Speed > 1.0 makes the waveform shorter.
        perturbed_samples = max(
            1,
            int(round(original_samples / float(speed_factor)))
        )

        output_frames = output_frames_for_samples(perturbed_samples)

        labels = example["labels"]
        min_frames = minimum_ctc_frames(labels)

        if output_frames < min_frames:
            bad.append({
                "index": i,
                "segment_id": example["segment_id"],
                "speed_factor": speed_factor,
                "original_samples": original_samples,
                "perturbed_samples": perturbed_samples,
                "perturbed_seconds": perturbed_samples / 16000.0,
                "output_frames": output_frames,
                "label_length": len(labels),
                "minimum_ctc_frames": min_frames,
            })

    return bad


bad_train_original = find_ctc_infeasible(train_ds, speed_factor=1.0)
bad_val = find_ctc_infeasible(val_ds, speed_factor=1.0)

# 1.1× is the shortest waveform in the planned speed set and therefore
# the strictest CTC-feasibility case.
bad_train_fast = find_ctc_infeasible(train_ds, speed_factor=1.1)

print("CTC-infeasible train at 1.0×:", len(bad_train_original))
print("CTC-infeasible train at 1.1×:", len(bad_train_fast))
print("CTC-infeasible clean validation:", len(bad_val))

if bad_train_original:
    display(pd.DataFrame(bad_train_original))
if bad_train_fast:
    display(pd.DataFrame(bad_train_fast))
if bad_val:
    display(pd.DataFrame(bad_val))

assert not bad_train_original, "Original training set contains CTC-infeasible examples."
assert not bad_train_fast, (
    "Some training examples become CTC-infeasible at 1.1×. "
    "Do not start training until the speed set is adjusted."
)
assert not bad_val, "Validation contains CTC-infeasible examples."

print("✓ Original, 1.1× training, and clean validation are all CTC-feasible.")


CTC-infeasible train at 1.0×: 0
CTC-infeasible train at 1.1×: 0
CTC-infeasible clean validation: 0
✓ Original, 1.1× training, and clean validation are all CTC-feasible.


In [ ]:
# Cell 14 — Define dynamic CTC padding and training-only speed perturbation

from dataclasses import dataclass
from typing import Dict, List, Union
from torch.utils.data import Dataset
import torch
import torch.nn.functional as F
import random


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        batch["labels"] = labels
        return batch


class SpeedPerturbedTrainingDataset(Dataset):
    """Apply speed perturbation only when a training example is fetched.

    A factor > 1.0 makes speech faster/shorter.
    A factor < 1.0 makes speech slower/longer.
    Labels are unchanged.

    Validation is NOT wrapped by this class.
    """

    def __init__(self, base_dataset, factors=(0.9, 1.0, 1.1)):
        self.base_dataset = base_dataset
        self.factors = tuple(float(x) for x in factors)

    def __len__(self):
        return len(self.base_dataset)

    @staticmethod
    def _change_speed(input_values, factor):
        waveform = torch.as_tensor(
            input_values,
            dtype=torch.float32,
        )

        if factor == 1.0:
            return waveform.tolist()

        original_length = waveform.numel()
        new_length = max(
            1,
            int(round(original_length / factor))
        )

        perturbed = F.interpolate(
            waveform.view(1, 1, -1),
            size=new_length,
            mode="linear",
            align_corners=False,
        ).view(-1)

        return perturbed.tolist()

    def __getitem__(self, index):
        example = dict(self.base_dataset[index])

        factor = random.choice(self.factors)

        example["input_values"] = self._change_speed(
            example["input_values"],
            factor,
        )
        example["input_length"] = len(example["input_values"])
        example["speed_factor"] = factor

        return example


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True,
)

train_speed_ds = SpeedPerturbedTrainingDataset(
    train_ds,
    factors=SPEED_PERTURBATION_CONFIG["factors"],
)

print("Original train examples:", len(train_ds))
print("Speed-perturbed train examples per epoch:", len(train_speed_ds))
print("Validation examples:", len(val_ds))
print("Speed factors:", SPEED_PERTURBATION_CONFIG["factors"])
print("✓ Speed perturbation is training-only; validation stays clean.")


Original train examples: 1754
Speed-perturbed train examples per epoch: 1754
Validation examples: 129
Speed factors: [0.9, 1.0, 1.1]
✓ Speed perturbation is training-only; validation stays clean.


In [ ]:
# Cell 15 — Define validation WER and CER
from jiwer import wer, cer


def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids)
    label_str = tokenizer.batch_decode(
        label_ids,
        group_tokens=False,
    )

    return {
        "wer": wer(label_str, pred_str),
        "cer": cer(label_str, pred_str),
    }

print("✓ WER/CER metric function ready.")


✓ WER/CER metric function ready.


In [ ]:
# Cell 16 — Set the same seed and define the best-adapter callback

import random
from transformers import TrainerCallback
from safetensors.torch import save_file as safe_save_file

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


COMBINED_AUGMENTATION_CONFIG = {
    "specaugment": SPEC_AUGMENT_CONFIG,
    "speed_perturbation": SPEED_PERTURBATION_CONFIG,
}


class BestAdapterCallback(TrainerCallback):
    def __init__(self, adapter_path, info_path):
        self.adapter_path = Path(adapter_path)
        self.info_path = Path(info_path)
        self.best_cer = float("inf")

        if self.info_path.exists():
            try:
                with open(self.info_path, "r", encoding="utf-8") as f:
                    previous = json.load(f)
                self.best_cer = float(previous.get("best_cer", float("inf")))
                print(f"Previous best CER recovered: {self.best_cer:.4f}")
            except Exception:
                pass

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        metrics = metrics or {}
        current_cer = metrics.get("eval_cer")

        if current_cer is None or model is None:
            return control

        if current_cer < self.best_cer:
            self.best_cer = float(current_cer)

            adapter_state = {
                name: tensor.detach().cpu().contiguous()
                for name, tensor in model._get_adapters().items()
            }

            safe_save_file(
                adapter_state,
                str(self.adapter_path),
                metadata={"format": "pt"},
            )

            info = {
                "best_cer": self.best_cer,
                "eval_wer": float(metrics.get("eval_wer", float("nan"))),
                "eval_loss": float(metrics.get("eval_loss", float("nan"))),
                "epoch": float(state.epoch) if state.epoch is not None else None,
                "global_step": int(state.global_step),
                "metadata_sha256": sha256,
                "base_model": BASE_MODEL_ID,
                "tokenizer_size": len(tokenizer),
                "augmentation": COMBINED_AUGMENTATION_CONFIG,
            }

            with open(self.info_path, "w", encoding="utf-8") as f:
                json.dump(info, f, indent=2)

            print(
                f"\n✓ New best SpecAugment + speed adapter saved — "
                f"CER={self.best_cer:.4f}, step={state.global_step}"
            )

        return control


best_adapter_callback = BestAdapterCallback(
    BEST_ADAPTER_PATH,
    BEST_ADAPTER_INFO_PATH,
)

print("✓ Reproducibility seed:", SEED)


✓ Reproducibility seed: 42


In [ ]:
# Cell 17 — Configure resumable SpecAugment + speed training

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(TRAINER_STATE_DIR),
    num_train_epochs=4,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=1e-3,
    warmup_steps=50,
    lr_scheduler_type="linear",

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",

    save_strategy="epoch",
    save_total_limit=1,

    logging_strategy="steps",
    logging_steps=25,

    report_to="none",
    seed=SEED,
    data_seed=SEED,

    # Keep augmentation RNG in the main process so checkpoint RNG state
    # can reproduce the training stream when resuming.
    dataloader_num_workers=0,

    remove_unused_columns=False,
)

print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Physical train batch:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print(
    "Effective batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)
print("Save strategy:", training_args.save_strategy)
print("Dataloader workers:", training_args.dataloader_num_workers)
print("✓ TrainingArguments ready.")


Epochs: 4
Learning rate: 0.001
Physical train batch: 2
Gradient accumulation: 8
Effective batch size: 16
Save strategy: SaveStrategy.EPOCH
Dataloader workers: 0
✓ TrainingArguments ready.


In [ ]:
# Cell 18 — Create the SpecAugment + speed Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=train_speed_ds,
    eval_dataset=val_ds,
    processing_class=processor.feature_extractor,
    callbacks=[best_adapter_callback],
)

print("Trainer train examples:", len(trainer.train_dataset))
print("Trainer validation examples:", len(trainer.eval_dataset))
print("SpecAugment enabled:", trainer.model.config.apply_spec_augment)
print("Speed factors:", SPEED_PERTURBATION_CONFIG["factors"])
print("Validation speed perturbation: False")

assert len(trainer.train_dataset) == 1754
assert len(trainer.eval_dataset) == 129
assert trainer.model.config.apply_spec_augment is True

print("✓ Combined-augmentation Trainer ready.")


Trainer train examples: 1754
Trainer validation examples: 129
SpecAugment enabled: True
Speed factors: [0.9, 1.0, 1.1]
Validation speed perturbation: False
✓ Combined-augmentation Trainer ready.


In [ ]:
# Cell 19 — Start or resume the V1.2 SpecAugment + speed run

from transformers.trainer_utils import get_last_checkpoint

RESUME_IF_AVAILABLE = True

last_checkpoint = None
if TRAINER_STATE_DIR.exists():
    last_checkpoint = get_last_checkpoint(str(TRAINER_STATE_DIR))

if RESUME_IF_AVAILABLE and last_checkpoint is not None:
    print("Resuming from:", last_checkpoint)
    train_result = trainer.train(
        resume_from_checkpoint=last_checkpoint
    )
else:
    print("Starting a fresh 4-epoch SpecAugment + speed-perturbation run.")
    train_result = trainer.train()

print("\nTraining complete.")
print(train_result)
print("\nBest adapter path:", BEST_ADAPTER_PATH)
print("Best adapter info:", BEST_ADAPTER_INFO_PATH)


Starting a fresh 4-epoch SpecAugment + speed-perturbation run.


Epoch,Training Loss,Validation Loss,Wer,Cer
1,0.730900,2.943674,0.884354,0.443524
2,0.662700,2.989371,0.866071,0.437736
3,0.598000,2.882991,0.870748,0.435405
4,0.599300,2.975424,0.856293,0.434360



✓ New best SpecAugment + speed adapter saved — CER=0.4435, step=110

✓ New best SpecAugment + speed adapter saved — CER=0.4377, step=220

✓ New best SpecAugment + speed adapter saved — CER=0.4354, step=330

✓ New best SpecAugment + speed adapter saved — CER=0.4344, step=440

Training complete.
TrainOutput(global_step=440, training_loss=1.2699071754108775, metrics={'train_runtime': 4196.1903, 'train_samples_per_second': 1.672, 'train_steps_per_second': 0.105, 'total_flos': 8.673818115787743e+18, 'train_loss': 1.2699071754108775, 'epoch': 4.0})

Best adapter path: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_specaug_speed/best_adapter.safetensors
Best adapter info: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_specaug_speed/best_adapter_info.json


In [ ]:
# Cell 20 — Reload the best SpecAugment + speed adapter

from safetensors.torch import load_file as safe_load_file

assert BEST_ADAPTER_PATH.exists(), "Best combined-augmentation adapter was not saved."

best_adapter_state = safe_load_file(
    str(BEST_ADAPTER_PATH),
    device="cpu",
)

load_result = model.load_state_dict(
    best_adapter_state,
    strict=False,
)

print("Adapter tensors loaded:", len(best_adapter_state))
print("Unexpected keys:", len(load_result.unexpected_keys))

assert len(load_result.unexpected_keys) == 0, (
    f"Unexpected adapter keys: {load_result.unexpected_keys[:20]}"
)

model.to(training_args.device)

with open(BEST_ADAPTER_INFO_PATH, "r", encoding="utf-8") as f:
    best_info = json.load(f)

print("\nBest combined-augmentation checkpoint information:")
print(json.dumps(best_info, indent=2))


Adapter tensors loaded: 290
Unexpected keys: 0

Best combined-augmentation checkpoint information:
{
  "best_cer": 0.4343596752150494,
  "eval_wer": 0.8562925170068028,
  "eval_loss": 2.975424289703369,
  "epoch": 4.0,
  "global_step": 440,
  "metadata_sha256": "4911f46e0a8c3656089677b8899d8296b9e1528d8aa3633a64728eb645f28fab",
  "base_model": "facebook/mms-1b-all",
  "tokenizer_size": 34,
  "augmentation": {
    "specaugment": {
      "apply_spec_augment": true,
      "mask_time_prob": 0.05,
      "mask_time_length": 5,
      "mask_time_min_masks": 1,
      "mask_feature_prob": 0.0
    },
    "speed_perturbation": {
      "enabled": true,
      "factors": [
        0.9,
        1.0,
        1.1
      ],
      "sampling": "uniform_random_per_training_example_access",
      "validation_augmented": false
    }
  }
}


In [ ]:
# Cell 21 — Evaluate the best combined-augmentation adapter on clean validation

validation_metrics = trainer.evaluate(
    eval_dataset=val_ds,
    metric_key_prefix="validation",
)

print("Best SpecAugment + speed adapter — clean validation metrics:")
for key, value in validation_metrics.items():
    print(f"{key}: {value}")


Best SpecAugment + speed adapter — clean validation metrics:
validation_loss: 2.975424289703369
validation_wer: 0.8562925170068028
validation_cer: 0.4343596752150494
validation_runtime: 22.0216
validation_samples_per_second: 5.858
validation_steps_per_second: 2.952
epoch: 4.0


In [ ]:
# Cell 22 — Save clean-validation predictions for qualitative analysis

prediction_output = trainer.predict(
    val_ds,
    metric_key_prefix="validation_prediction",
)

pred_ids = np.argmax(
    prediction_output.predictions,
    axis=-1,
)

pred_texts = tokenizer.batch_decode(pred_ids)

reference_texts = [
    tokenizer.decode(
        example["labels"],
        group_tokens=False,
    )
    for example in val_ds
]

validation_predictions = pd.DataFrame({
    "segment_id": val_ds["segment_id"],
    "reference": reference_texts,
    "prediction": pred_texts,
})

PREDICTIONS_PATH = RESULTS_DIR / "validation_predictions.csv"

validation_predictions.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8",
)

display(validation_predictions.head(20))
print("\nSaved:", PREDICTIONS_PATH)


,segment_id,reference,prediction
0,REC090_SEG0010,ssalamuɛlikum necc meryem,ssalamuɛlikum nec meryam
1,REC090_SEG0011,aqay ruxxa tnayn uɛecrin sana ḍi hulanda,aqay ruxxa tnayn uɛecrin sanad ihulanḍa
2,REC090_SEG0012,mercex ak nmis n jjiran usiɣ ḍ zi lmeɣrib umi ...,mercex ag misen jjiran usiɣ dzi lmeɣrib umiraq...
3,REC090_SEG0013,umi wsiɣd dda ufix manayenni wa dji ca min ira...,umi wsiɣ dda ufi x manayenni wadji ca mirira ɣ...
4,REC090_SEG0014,a necc mammec ira djjix ḍi lmeɣrib wadji manay...,anecc mamci ra dji x di lmeɣrib wadji man ayen...
5,REC090_SEG0015,necc ḍi lmeɣrib ira ɣari lḥurriya inu ira ɣari...,necddi lmeɣrib ira ɣari lḥurri ainu ira ɣari i...
6,REC090_SEG0016,ḍi lmeɣrib neccin mammec ira niɛicc,ḍi lmeɣrim nccin mamcirani ɛica
7,REC090_SEG0017,ak baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥway...,ag babadimma wa ɣanex ca ɣanex caneḥwayej nteg...
8,REC090_SEG0018,lmuhim wsiɣd,muhim wsiɣt
9,REC090_SEG0019,necc ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ḍi ṭi...,neccira ɛemmas wa wsiɣ daɣa uruppa wsiɣ ddi ṭi...



Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_1b_tarifit_v1_2_specaug_speed/validation_predictions.csv


In [ ]:
# Cell 23 — Save training history and the combined-augmentation experiment summary

history_df = pd.DataFrame(trainer.state.log_history)
HISTORY_PATH = RESULTS_DIR / "training_history.csv"
history_df.to_csv(HISTORY_PATH, index=False)

processor.save_pretrained(MODEL_DIR / "processor")
model.config.save_pretrained(MODEL_DIR / "config")

summary = {
    "experiment": "MMS-1B Tarifit V1.2 adapter fine-tuning — SpecAugment + speed perturbation",
    "base_model": BASE_MODEL_ID,
    "metadata_path": str(FROZEN_METADATA_PATH),
    "metadata_sha256": sha256,
    "train_segments": len(train_ds),
    "validation_segments": len(val_ds),
    "train_hours_original": train_duration["hours"],
    "validation_hours": val_duration["hours"],
    "tokenizer_size": len(tokenizer),
    "final_letters": FINAL_LETTERS,
    "seed": SEED,
    "num_train_epochs": 4,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "effective_train_batch_size": 16,
    "learning_rate": 1e-3,
    "warmup_steps": 50,
    "augmentation": COMBINED_AUGMENTATION_CONFIG,
    "best_adapter": best_info,
    "final_validation_metrics": {
        key: float(value) if isinstance(value, (int, float, np.floating)) else value
        for key, value in validation_metrics.items()
    },
}

SUMMARY_PATH = RESULTS_DIR / "experiment_summary.json"

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved training history:", HISTORY_PATH)
print("Saved experiment summary:", SUMMARY_PATH)
print("Saved processor:", MODEL_DIR / "processor")
print("Saved best adapter:", BEST_ADAPTER_PATH)


Saved training history: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_1b_tarifit_v1_2_specaug_speed/training_history.csv
Saved experiment summary: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_1b_tarifit_v1_2_specaug_speed/experiment_summary.json
Saved processor: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_specaug_speed/processor
Saved best adapter: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_specaug_speed/best_adapter.safetensors


In [ ]:
# Cell 24 — Compare all three controlled V1.2 MMS experiments

def read_summary(path):
    if not path.exists():
        return None

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


noaug_summary = read_summary(NOAUG_SUMMARY_PATH)
specaug_summary = read_summary(SPECAUG_SUMMARY_PATH)
combined_summary = read_summary(SUMMARY_PATH)

rows = []

for label, experiment_summary in [
    ("V1.2 no augmentation", noaug_summary),
    ("V1.2 SpecAugment", specaug_summary),
    ("V1.2 SpecAugment + speed", combined_summary),
]:
    if experiment_summary is None:
        continue

    best = experiment_summary.get("best_adapter", {})

    rows.append({
        "experiment": label,
        "best_epoch": best.get("epoch"),
        "WER": best.get("eval_wer"),
        "CER": best.get("best_cer"),
    })


comparison = pd.DataFrame(rows)
display(comparison)

if len(comparison) >= 2:
    print("\nLower WER/CER is better.")

if specaug_summary is None:
    print("\nSpecAugment summary not found:", SPECAUG_SUMMARY_PATH)

if noaug_summary is None:
    print("\nNo-augmentation summary not found:", NOAUG_SUMMARY_PATH)


,experiment,best_epoch,WER,CER
0,V1.2 no augmentation,3.0,0.866497,0.434199
1,V1.2 SpecAugment,3.0,0.859694,0.432671
2,V1.2 SpecAugment + speed,4.0,0.856293,0.434360



Lower WER/CER is better.


## Interpretation rule

Use validation only for model selection and controlled comparison.

The final test set remains untouched until its manual transcription and selection are frozen.

The main controlled comparison is:

1. MMS V1.2 — no augmentation
2. MMS V1.2 — SpecAugment
3. MMS V1.2 — SpecAugment + random speed perturbation (0.9× / 1.0× / 1.1×)

Only after choosing the final system(s) from validation should the held-out test set be evaluated.


SpecAugment produced a small but consistent improvement over training without augmentation. Combining SpecAugment with speed perturbation yielded the lowest WER (85.63%), although this improvement was not reflected in CER, which remained comparable to the non-augmented configuration.




